# SOAR-RAG: Self-Optimizing Attention-Guided Retrieval-Augmented Generation
Runnable end-to-end on **Colab (free T4 GPU)** or **Kaggle**.

This notebook: installs deps, clones your repo (or uses the uploaded `soar-rag/` folder),
downloads an open-weight LLM, builds a knowledge base, runs a live demo query showing the
iterative retrieval->generation->attention-feedback->rerank loop, then runs the full
4-dataset evaluation (NQ, HotpotQA, TriviaQA, BioASQ) comparing baseline RAG vs SOAR-RAG.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is enough for a 1.5B-3B model).

In [ ]:
!pip install -q torch transformers sentence-transformers faiss-cpu datasets fastapi uvicorn pydantic pandas numpy matplotlib
# If running in Colab, upload/clone your soar-rag project folder so `src/`, `eval/`, `data/` are importable:
# from google.colab import drive; drive.mount('/content/drive')
# !git clone <your-github-repo-url> soar-rag
%cd soar-rag
import sys; sys.path.insert(0, '.')

In [ ]:
from src.knowledge_base import KnowledgeBase
from src.generation import LLMGenerator
from src.pipeline import SOARRAGPipeline, BaselineRAGPipeline

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # bump to a 7B model if you have an A100/L4
generator = LLMGenerator(model_name=MODEL_NAME, device="cuda")
print("Model loaded:", MODEL_NAME)

## 1. Quick live demo on a toy knowledge base

In [ ]:
demo_corpus = [
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, completed in 1889.",
    "Paris is the capital and most populous city of France, known for its art, fashion, and culture.",
    "The Great Wall of China is a series of fortifications built across northern China.",
    "Photosynthesis is the process by which green plants convert sunlight into chemical energy.",
    "Mount Everest, located in the Himalayas, is Earth's highest mountain above sea level.",
]
kb = KnowledgeBase(device="cuda")
kb.build(demo_corpus)

soar = SOARRAGPipeline(kb, generator, max_iterations=3)
result = soar.run("When was the Eiffel Tower completed and where is it located?", verbose=True)

print("\nFinal answer:", result.final_response)
print("Stopped because:", result.converged_reason)
print("Context compression ratio:", round(result.context_compression_ratio, 3))
for t in result.trace:
    print(f"  iter {t.iteration}: k={t.top_k} conf={t.confidence:.3f} entropy={t.attention_entropy:.3f} docs={t.retrieved_doc_ids}")

## 2. Compare against a plain baseline RAG (single retrieval, no feedback)

In [ ]:
baseline = BaselineRAGPipeline(kb, generator, top_k=5)
b_result = baseline.run("When was the Eiffel Tower completed and where is it located?")
print("Baseline answer:", b_result.final_response)
print("SOAR-RAG answer:", result.final_response)

## 3. Full evaluation: 4 datasets, all paper metrics
Downloads NQ-open, HotpotQA, TriviaQA and (PubMedQA as a BioASQ stand-in, or your own
BioASQ export) subsets, builds a per-dataset knowledge base from their contexts, and runs
baseline RAG vs SOAR-RAG computing EM, F1, retrieval accuracy, hallucination rate, context
compression ratio, and latency for both.

Start with `n=30` per dataset to sanity-check timing on a T4 before scaling up to `n=100-300`.

In [ ]:
!python -m eval.run_eval --datasets nq hotpotqa triviaqa bioasq --n 30 --model Qwen/Qwen2.5-1.5B-Instruct --out results

In [ ]:
import pandas as pd
summary = pd.read_csv("results/summary.csv", index_col=0)
summary

## 4. Plot baseline vs SOAR-RAG across datasets

In [ ]:
import matplotlib.pyplot as plt

metrics = [("EM", "baseline_EM", "soar_EM"), ("F1", "baseline_F1", "soar_F1"),
           ("RetrievalAcc", "baseline_RetrievalAcc", "soar_RetrievalAcc"),
           ("HallucinationRate", "baseline_HallucinationRate", "soar_HallucinationRate")]

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 4))
for ax, (title, bcol, scol) in zip(axes, metrics):
    summary[[bcol, scol]].plot(kind="bar", ax=ax, legend=(title == "EM"))
    ax.set_title(title)
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig("results/comparison.png", dpi=150)
plt.show()

## 5. Save artifacts for your resume/portfolio
- `results/summary.csv`, `results/comparison.png` -> put these in your README / project report.
- Push this repo (with `results/`) to GitHub.
- Optionally deploy `app/api.py` (FastAPI) to a small GPU host (e.g. a Colab-backed ngrok tunnel
  for a demo link, or HuggingFace Spaces with a GPU Space) so you can share a live link.